In [41]:
# NER Token PCA Analysis 
import sys
from pathlib import Path
import json
import numpy as np
import torch
from sklearn.decomposition import PCA
from collections import Counter

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

In [42]:
# Load pre-computed artifacts
output_dir = repo_root / "output" / "ner_conll2003_bert"
embeddings_dir = output_dir / "token_layer_embeddings" / "train" / "last"
index_path = output_dir / "indexes" / "train" / "last" / "euclidean" / "annoy" / "annoy_index.ann"

# Load embeddings and metadata
data = np.load(embeddings_dir / "token_vectors.npz")
train_embeddings = data["vectors"].astype(np.float32)
with open(embeddings_dir / "token_metadata.json", 'r') as f:
    metadata = json.load(f)
train_tokens = metadata["token_texts"]
train_label_ids = np.array(metadata["label_ids"], dtype=np.int64)

# Load Annoy index
import annoy
knn_index = annoy.AnnoyIndex(train_embeddings.shape[1], 'euclidean')
knn_index.load(str(index_path))

print(f"Loaded {train_embeddings.shape[0]:,} token embeddings ({train_embeddings.shape[1]} dim)")

Loaded 203,621 token embeddings (768 dim)


In [43]:
# Load classifier weights
model_path = output_dir / "ner_bert_conll2003_finetune" / "model.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_dict = torch.load(str(model_path), map_location=device)
classifier_weight = state_dict["classifier.weight"].to(device)
classifier_bias = state_dict["classifier.bias"].to(device)

# Label definitions
LABEL_NAMES = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
def label_name(lid): return LABEL_NAMES[lid] if 0 <= lid < len(LABEL_NAMES) else f"UNK-{lid}"
def entity_type(lid): 
    name = label_name(lid)
    return "O" if name == "O" else name.split("-")[1] if "-" in name else name

print(f"Classifier loaded: {classifier_weight.shape[0]} labels, {classifier_weight.shape[1]} dim")

Classifier loaded: 9 labels, 768 dim


In [44]:
# Core functions
def to_vec(x): return np.asarray(x, dtype=np.float32).reshape(-1)

def classify(h):
    """Classify hidden state → (pred_label, confidence, probs)"""
    with torch.no_grad():
        logits = torch.from_numpy(to_vec(h)).to(device) @ classifier_weight.T + classifier_bias
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    pred_id = int(np.argmax(probs))
    return label_name(pred_id), float(probs[pred_id]), probs

def get_neighbors(vec, k=10):
    """Get k nearest neighbors"""
    ids = knn_index.get_nns_by_vector(to_vec(vec).tolist(), k)
    return [(i, train_tokens[i], label_name(train_label_ids[i])) for i in ids]

def whiten(x, Vt, ev): return (to_vec(x) @ Vt.T) / np.sqrt(ev)
def unwhiten(w, mean, Vt, ev): return (to_vec(w) * np.sqrt(ev)) @ Vt + mean

In [45]:
# Pick one token from each major entity category
def find_first_token(label_id):
    idx = int(np.where(train_label_ids == label_id)[0][0])
    return idx, train_tokens[idx]

tokens = {
    "LOC": find_first_token(5),   # B-LOC
    "PER": find_first_token(1),   # B-PER  
    "ORG": find_first_token(3),   # B-ORG
}
print("Selected tokens:")
for cat, (idx, tok) in tokens.items():
    print(f"  {cat}: idx={idx}, '{tok}'")

Selected tokens:
  LOC: idx=11, 'brussels'
  PER: idx=9, 'peter'
  ORG: idx=0, 'eu'


In [63]:
def analyze_token(target_idx, k_neighbors=200, n_pcs=10, steps=[-3,-2,-1,0,1,2,3]):
    """Analyze one token: neighbors, PCA, reconstruction table"""
    
    target_tok = train_tokens[target_idx]
    target_label = label_name(train_label_ids[target_idx])
    target_h = train_embeddings[target_idx]
    orig_pred, orig_conf, _ = classify(target_h)
    
    print("=" * 80)
    print(f"TARGET: '{target_tok}' | Original Label: {target_label} | Pred: {orig_pred} ({orig_conf:.0%})")
    print("=" * 80)
    
    # 1. Show neighbors
    print(f"\n--- Top 35 Neighbors Embeddings ---")
    neighbors = get_neighbors(target_h, 35)
    for i, (nid, tok, lbl) in enumerate(neighbors):
        same = "Same token different represenation" if tok == target_tok else ""
        print(f"  {i+1:2d}. idx={nid:6d} '{tok}' ({lbl}) {same}")
    
    # 2. Fit local PCA
    neighbor_ids = knn_index.get_nns_by_item(target_idx, k_neighbors)
    local_h = train_embeddings[neighbor_ids]
    mean = local_h.mean(axis=0).astype(np.float32)
    pca = PCA(n_components=64)
    pca.fit(local_h - mean)
    Vt = pca.components_.astype(np.float32)
    ev = np.maximum(pca.explained_variance_.astype(np.float32), 1e-8)
    
    print(f"\n--- Local PCA ({k_neighbors} neighbors) ---")
    print(f"Variance explained: {pca.explained_variance_ratio_[:5]}")
    
    # 3. Analyze each PC direction
    target_w = whiten(target_h, Vt, ev)
    gold_label_id = train_label_ids[target_idx]
    
    for pc in range(n_pcs):
        print(f"\n--- PC{pc} (var={pca.explained_variance_ratio_[pc]:.1%}) ---")
        print(f"{'Step':>5} | {'h_noisy (Pertub state) Classfication':^18} | {'Nearest Token in Neighbourhood':^20} | {'Org Label':^8} | {'nearest_to_h_noisy_class':^18}")
        print("-" * 90)
        
        for step in steps:
            w = target_w.copy()
            w[pc] += step
            h_noisy = unwhiten(w, mean, Vt, ev)
            
            # Classify noisy h directly
            noisy_pred, noisy_conf, noisy_probs = classify(h_noisy)
            noisy_ok = "✓" if noisy_pred == target_label else "✗"
            
            # Find nearest real token (reconstruction)
            nn_idx, nn_tok, nn_gold = get_neighbors(h_noisy, 1)[0]
            nn_pred, nn_conf, _ = classify(train_embeddings[nn_idx])
            nn_ok = "✓" if nn_pred == target_label else "✗"
            same = "-" if nn_tok == target_tok else " "
            
            print(f"{step:>+5} | {noisy_ok} {noisy_pred:>6} ({noisy_conf:>5.1%}) | {same}'{nn_tok[:16]:<16}' | {nn_gold:^8} | {nn_ok} {nn_pred:>6} ({nn_conf:>5.1%})")

In [78]:
def show_noisy_samples(target_idx, k_neighbors=200, n_samples=5, noise_scale=1.0):
    """NER noisy sampling visualization."""
    target_tok = train_tokens[target_idx]
    target_label = label_name(train_label_ids[target_idx])
    target_h = train_embeddings[target_idx]
    orig_pred, orig_conf, _ = classify(target_h)
    
    # Fit local PCA
    neighbor_ids = knn_index.get_nns_by_item(target_idx, k_neighbors)
    local_h = train_embeddings[neighbor_ids]
    mean = local_h.mean(axis=0).astype(np.float32)
    pca = PCA(n_components=64)
    pca.fit(local_h - mean)
    Vt = pca.components_.astype(np.float32)
    ev = np.maximum(pca.explained_variance_.astype(np.float32), 1e-8)
    target_w = whiten(target_h, Vt, ev)
    
    print(f"Target: '{target_tok}' | Original Label: {target_label} | Pred: {orig_pred} ({orig_conf:.0%})\n")
    
    # 1. Noisy whitened space
    print(f"Manifold noise in whitened space (σ={noise_scale}):")
    for i in range(n_samples):
        eps = np.random.normal(0, noise_scale, size=len(target_w)).astype(np.float32)
        h_noisy = unwhiten(target_w + eps, mean, Vt, ev)
        pred, conf, _ = classify(h_noisy)
        nn_idx, nn_tok, nn_lbl = get_neighbors(h_noisy, 1)[0]
        print(f"  {i+1}. Hiden Prediction: {pred}({conf:.0%}) | Nearest Neigbour of noisy represenation:'{nn_tok}'({nn_lbl})")
    
    # 2. Noisy hidden space
    print(f"\nGuassian noise in last layer hidden represenation (σ={noise_scale}):")
    for i in range(n_samples):
        eps = np.random.normal(0, noise_scale, size=len(target_h)).astype(np.float32)
        h_noisy = target_h + eps
        pred, conf, _ = classify(h_noisy)
        nn_idx, nn_tok, nn_lbl = get_neighbors(h_noisy, 1)[0]
        print(f"  {i+1}. Hiden Prediction: {pred}({conf:.0%}) | Nearest Neigbour of noisy represenation:'{nn_tok}'({nn_lbl})")
    
    # 3. Neighbors
    print(f"\n Neighbours:")
    for i, nid in enumerate(neighbor_ids[:n_samples]):
        tok, lbl = train_tokens[nid], label_name(train_label_ids[nid])
        pred, conf, _ = classify(train_embeddings[nid])
        print(f"  {i+1}. '{tok}' | Origianl Label:{lbl} | Pred:{pred}({conf:.0%})")

In [79]:
# Show noisy samples for B-LOC token (like CelebA visualization)
show_noisy_samples(tokens["LOC"][0], n_samples=5, noise_scale=1.0)

Target: 'brussels' | Original Label: B-LOC | Pred: B-LOC (100%)

Manifold noise in whitened space (σ=1.0):
  1. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'brussels'(B-LOC)
  2. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'brussels'(B-LOC)
  3. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'brussels'(B-LOC)
  4. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'brussels'(B-LOC)
  5. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'brussels'(B-LOC)

Guassian noise in last layer hidden represenation (σ=1.0):
  1. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'brussels'(B-LOC)
  2. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'brussels'(B-LOC)
  3. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represenation:'copenhagen'(B-LOC)
  4. Hiden Prediction: B-LOC(100%) | Nearest Neigbour of noisy represen

In [80]:
# Show noisy samples for B-PER token
show_noisy_samples(tokens["PER"][0], n_samples=5, noise_scale=1.0)

Target: 'peter' | Original Label: B-PER | Pred: B-PER (100%)

Manifold noise in whitened space (σ=1.0):
  1. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'peter'(B-PER)
  2. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'peter'(B-PER)
  3. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'peter'(B-PER)
  4. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'peter'(B-PER)
  5. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'gary'(B-PER)

Guassian noise in last layer hidden represenation (σ=1.0):
  1. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'arjun'(B-PER)
  2. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'peter'(B-PER)
  3. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'steve'(B-PER)
  4. Hiden Prediction: B-PER(100%) | Nearest Neigbour of noisy represenation:'craig'(B-PER)
  5. Hide

# Single signle PCA direction Movement

In [64]:
# Analyze B-LOC token
analyze_token(tokens["LOC"][0])

TARGET: 'brussels' | Original Label: B-LOC | Pred: B-LOC (100%)

--- Top 35 Neighbors Embeddings ---
   1. idx=    11 'brussels' (B-LOC) Same token different represenation
   2. idx= 11290 'brussels' (B-LOC) Same token different represenation
   3. idx= 32663 'brussels' (B-LOC) Same token different represenation
   4. idx= 32329 'brussels' (B-LOC) Same token different represenation
   5. idx= 31952 'brussels' (B-LOC) Same token different represenation
   6. idx= 32270 'brussels' (B-LOC) Same token different represenation
   7. idx= 33266 'brussels' (B-LOC) Same token different represenation
   8. idx= 31733 'brussels' (B-LOC) Same token different represenation
   9. idx= 74081 'brussels' (B-LOC) Same token different represenation
  10. idx= 54374 'brussels' (B-LOC) Same token different represenation
  11. idx= 61456 'brussels' (B-LOC) Same token different represenation
  12. idx=173423 'brussels' (B-LOC) Same token different represenation
  13. idx=197018 'brussels' (B-LOC) Same token 

In [69]:
analyze_token(tokens["LOC"][0], k_neighbors=200, n_pcs=10, steps=[-15,-10,-5,0,5,10,15])

TARGET: 'brussels' | Original Label: B-LOC | Pred: B-LOC (100%)

--- Top 35 Neighbors Embeddings ---
   1. idx=    11 'brussels' (B-LOC) Same token different represenation
   2. idx= 11290 'brussels' (B-LOC) Same token different represenation
   3. idx= 32663 'brussels' (B-LOC) Same token different represenation
   4. idx= 32329 'brussels' (B-LOC) Same token different represenation
   5. idx= 31952 'brussels' (B-LOC) Same token different represenation
   6. idx= 32270 'brussels' (B-LOC) Same token different represenation
   7. idx= 33266 'brussels' (B-LOC) Same token different represenation
   8. idx= 31733 'brussels' (B-LOC) Same token different represenation
   9. idx= 74081 'brussels' (B-LOC) Same token different represenation
  10. idx= 54374 'brussels' (B-LOC) Same token different represenation
  11. idx= 61456 'brussels' (B-LOC) Same token different represenation
  12. idx=173423 'brussels' (B-LOC) Same token different represenation
  13. idx=197018 'brussels' (B-LOC) Same token 

In [68]:
analyze_token(tokens["PER"][0], k_neighbors=200, n_pcs=10, steps=[-15,-10,-5,0,5,10,15]) 

TARGET: 'peter' | Original Label: B-PER | Pred: B-PER (100%)

--- Top 35 Neighbors Embeddings ---
   1. idx=     9 'peter' (B-PER) Same token different represenation
   2. idx=163833 'peter' (B-PER) Same token different represenation
   3. idx=111940 'ian' (B-PER) 
   4. idx=119139 'peter' (B-PER) Same token different represenation
   5. idx= 73811 'peter' (B-PER) Same token different represenation
   6. idx=111892 'john' (B-PER) 
   7. idx= 45624 'steve' (B-PER) 
   8. idx=154305 'peter' (B-PER) Same token different represenation
   9. idx=122523 'peter' (B-PER) Same token different represenation
  10. idx=111924 'graham' (B-PER) 
  11. idx=181723 'peter' (B-PER) Same token different represenation
  12. idx= 46341 'peter' (B-PER) Same token different represenation
  13. idx= 73246 'peter' (B-PER) Same token different represenation
  14. idx= 73805 'gary' (B-PER) 
  15. idx=138901 'roger' (B-PER) 
  16. idx=  5452 'tom' (B-PER) 
  17. idx= 24233 'bernard' (B-PER) 
  18. idx=138904 'mar

In [65]:
# Analyze B-PER token
analyze_token(tokens["PER"][0])

TARGET: 'peter' | Original Label: B-PER | Pred: B-PER (100%)

--- Top 35 Neighbors Embeddings ---
   1. idx=     9 'peter' (B-PER) Same token different represenation
   2. idx=163833 'peter' (B-PER) Same token different represenation
   3. idx=111940 'ian' (B-PER) 
   4. idx=119139 'peter' (B-PER) Same token different represenation
   5. idx= 73811 'peter' (B-PER) Same token different represenation
   6. idx=111892 'john' (B-PER) 
   7. idx= 45624 'steve' (B-PER) 
   8. idx=154305 'peter' (B-PER) Same token different represenation
   9. idx=122523 'peter' (B-PER) Same token different represenation
  10. idx=111924 'graham' (B-PER) 
  11. idx=181723 'peter' (B-PER) Same token different represenation
  12. idx= 46341 'peter' (B-PER) Same token different represenation
  13. idx= 73246 'peter' (B-PER) Same token different represenation
  14. idx= 73805 'gary' (B-PER) 
  15. idx=138901 'roger' (B-PER) 
  16. idx=  5452 'tom' (B-PER) 
  17. idx= 24233 'bernard' (B-PER) 
  18. idx=138904 'mar

In [66]:
# Analyze B-ORG token
analyze_token(tokens["ORG"][0])

TARGET: 'eu' | Original Label: B-ORG | Pred: B-ORG (100%)

--- Top 35 Neighbors Embeddings ---
   1. idx=     0 'eu' (B-ORG) Same token different represenation
   2. idx=188984 'eu' (B-ORG) Same token different represenation
   3. idx=  5216 'eu' (B-ORG) Same token different represenation
   4. idx=187153 'eu' (B-ORG) Same token different represenation
   5. idx=   249 'eu' (B-ORG) Same token different represenation
   6. idx=189131 'eu' (B-ORG) Same token different represenation
   7. idx=188932 'eu' (B-ORG) Same token different represenation
   8. idx=   209 'eu' (B-ORG) Same token different represenation
   9. idx=  5206 'eu' (B-ORG) Same token different represenation
  10. idx=181416 'nato' (B-ORG) 
  11. idx=   275 'eu' (B-ORG) Same token different represenation
  12. idx=181493 'nato' (B-ORG) 
  13. idx=  5109 'eu' (B-ORG) Same token different represenation
  14. idx=173853 'eu' (B-ORG) Same token different represenation
  15. idx=  5082 'eu' (B-ORG) Same token different represen